# Numerical Inverse Kinematics (IK) — Math Notes (Matches Current Code)

This document describes the math implemented in the current code:

- `kinematics.py`: Denavit–Hartenberg forward kinematics `forwardKinematicsT`
- `planner.py`: rotation vectors, pose error, finite-difference Jacobian, numerical IK, trajectory generation
- `constraints.py`: joint-limit clamping
- `main.py`: pose target generation + Jacobian conditioning check

---

## 1) Homogeneous transforms

The end-effector pose is represented as a homogeneous transform

$$
T(q)=
\begin{bmatrix}
R(q) & p(q)\\
0 & 1
\end{bmatrix},
\quad R\in SO(3),\; p\in\mathbb{R}^3.
$$

- $p(q)$ is the end-effector position in the base frame.
- $R(q)$ is the end-effector rotation matrix in the base frame.

---

## 2) DH Forward kinematics (`forwardKinematicsT`)

Each joint transform is built from DH parameters $(a_i,\alpha_i,d_i,\theta_i)$:

$$
A_i(a_i,\alpha_i,d_i,\theta_i)=
\begin{bmatrix}
\cos\theta_i & -\sin\theta_i\cos\alpha_i & \sin\theta_i\sin\alpha_i & a_i\cos\theta_i \\
\sin\theta_i & \cos\theta_i\cos\alpha_i  & -\cos\theta_i\sin\alpha_i & a_i\sin\theta_i \\
0            & \sin\alpha_i               & \cos\alpha_i              & d_i \\
0            & 0                          & 0                         & 1
\end{bmatrix}.
$$

The full FK is the product:

$$
T_{0,6}(q)=A_1(q_1)A_2(q_2)A_3(q_3)A_4(q_4)A_5(q_5)A_6(q_6).
$$

The code uses UR5-like constants (meters):
$d_1=0.089$, $a_2=-0.425$, $a_3=-0.392$, $d_4=0.109$, $d_5=0.095$, $d_6=0.082$.

When `returnPoints=True`, FK also returns joint positions $p_0,\dots,p_6$ for visualization.

---

## 3) Rotation vectors (axis–angle)

Orientation targets are represented by a **rotation vector** $r\in\mathbb{R}^3$:

- $\theta=\|r\|$ is the rotation angle
- $k=r/\|r\|$ is the unit axis (for $\theta\neq 0$)
- $r=\theta k$

### 3.1 Rotation vector → rotation matrix (`vectorToR`)
Rodrigues’ formula:

$$
R(r)=I+\sin\theta [k]_\times + (1-\cos\theta)[k]_\times^2,
$$

with

$$
[k]_\times=
\begin{bmatrix}
0 & -k_z & k_y\\
k_z & 0 & -k_x\\
-k_y & k_x & 0
\end{bmatrix}.
$$

### 3.2 Rotation matrix → rotation vector (`RToVector`)
Compute:

$$
\theta = \cos^{-1}\left(\frac{\mathrm{tr}(R)-1}{2}\right).
$$

For $\theta\neq 0$:

$$
w = \frac{1}{2\sin\theta}
\begin{bmatrix}
R_{3,2}-R_{2,3}\\
R_{1,3}-R_{3,1}\\
R_{2,1}-R_{1,2}
\end{bmatrix},
\quad r=\theta w.
$$

---

## 4) Pose error used for IK (`poseError`)

A target pose is a 6-vector:

$$
\text{targets} = [x,\;y,\;z,\;r_x,\;r_y,\;r_z]^\top.
$$

Desired position and orientation:

$$
p_{\text{des}} = [x,y,z]^\top,\quad R_{\text{des}} = R(r).
$$

From FK we get current pose $(p_{\text{cur}},R_{\text{cur}})$.

### 4.1 Position error
$$
e_{\text{pos}} = p_{\text{des}} - p_{\text{cur}}.
$$

### 4.2 Orientation error (IMPORTANT: matches current code)
The code defines the relative rotation as **current → desired**:

$$
R_{\text{err}} = R_{\text{des}}\,R_{\text{cur}}^\top,
\quad
e_{\text{rot}} = \mathrm{rotvec}(R_{\text{err}}).
$$

This is exactly what is implemented as `RToVector(RDes @ RCur.T)`.

### 4.3 Weighted 6D pose error
The total 6D residual is:

$$
e(q)=
\begin{bmatrix}
w_{\text{pos}}\,e_{\text{pos}}\\
w_{\text{rot}}\,e_{\text{rot}}
\end{bmatrix}
\in\mathbb{R}^6.
$$

In the current code: $w_{\text{pos}}=1.0$, $w_{\text{rot}}=0.05$.

---

## 5) Numerical Jacobian (finite differences) (`Jacobianfd`)

The Jacobian is the sensitivity of the residual to joint angles:

$$
J(q)=\frac{\partial e}{\partial q}\in\mathbb{R}^{6\times 6}.
$$

It is approximated by forward differences:

$$
J_{:,i}\approx \frac{e(q+\varepsilon e_i)-e(q)}{\varepsilon}.
$$

This requires $n+1$ residual evaluations per iteration (here 7), and each residual evaluation calls FK.

---

## 6) Numerical IK step (`solvePoseIK`)

At each iteration, we solve a damped least-squares system with smoothness:

- Let $q_{\text{prev}}$ be the previous timestep solution (for continuity).
- The code forms:

$$
A = J^\top J + (\lambda + \mu)I,
$$

$$
b = -J^\top e - \mu (q - q_{\text{prev}}),
$$

and then solves

$$
A\,\Delta q = b,
\quad
q \leftarrow q + s\,\Delta q.
$$

Where:
- $\lambda$ is `damping`
- $\mu$ is `smoothw`
- $s$ is `stepScale`

Finally, joint limits are enforced by clamping:

$$
q_i \leftarrow \min(\max(q_i, q_i^{\min}), q_i^{\max}).
$$

Convergence stops when $\|e(q)\|<\text{tol}$ or after `maxIters`.

---

## 7) Trajectory generation (`generateTrajectoryPose`)

The target array has shape $(N,6)$ (one pose per waypoint). The planner solves IK sequentially:

1. initialize $q_0=q_{\text{start}}$
2. for each waypoint $i$:
   - solve $q_{i+1}=\text{IK}(q_i,\text{targets}[i])$
   - store $q_{i+1}$

Warm-starting each waypoint with the previous solution improves continuity.

In `main.py` the targets are created with a linear interpolation in 6D:

$$
\text{targets}[i] = (1-\alpha_i)\,\text{start} + \alpha_i\,\text{goal},
\quad \alpha_i \in [0,1].
$$

---

## 8) Jacobian conditioning diagnostic (singular values)

After planning, `main.py` computes SVD of the Jacobian:

$$
J = U\Sigma V^\top,
\quad \Sigma = \mathrm{diag}(\sigma_1,\dots,\sigma_6).
$$

The condition number is:

$$
\kappa(J)=\frac{\sigma_{\max}}{\sigma_{\min}}.
$$

A large $\kappa(J)$ indicates an ill-conditioned / near-singular configuration, which often causes larger Cartesian error for numerical IK.

---

## 9) Summary of the full pipeline

1. Choose $q_{\text{start}}$ and joint limits.
2. Compute `start` from FK and define `goal`.
3. Create a pose trajectory `targets` by linear interpolation.
4. For each waypoint, run numerical IK (finite-difference Jacobian + damped least squares + smoothness).
5. Visualize the resulting joint trajectory and optionally evaluate Jacobian conditioning.

---
